# Phase 1 - Modeling Layer, Metric Dictionary, Marts, SQL, NoSQL, Vector DB

**Intended use:** Build a Kimball-style warehouse and certified marts that Power BI and the app read (never bronze).

**Prerequisite:** Phase 0 (silver encounters exist).


## 1. Setup

Project root, `datafile.txt` registry helpers.


In [1]:
from __future__ import annotations
import json, hashlib, os, re, uuid, warnings, shutil
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path(".").resolve()
if not (ROOT / "datafile.txt").exists():
    ROOT = Path("..").resolve()
os.chdir(ROOT)
DATAFILE = ROOT / "datafile.txt"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def load_registry(path=DATAFILE):
    rows = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split("|")
        if len(parts) < 4:
            continue
        rows.append({"role": parts[0].strip(), "zone": parts[1].strip(), "path": parts[2].strip(), "description": parts[3].strip()})
    return pd.DataFrame(rows)

def registry_paths(role=None):
    reg = load_registry()
    if role is not None:
        reg = reg[reg["role"] == role]
    return [ROOT / p for p in reg["path"].tolist()]

def registry_path(role, default=None):
    paths = registry_paths(role=role)
    if paths:
        return paths[0]
    return ROOT / default if default else None

def upsert_registry(role, zone, rel_path, description):
    lines = DATAFILE.read_text(encoding="utf-8").splitlines()
    new_line = f"{role}|{zone}|{rel_path}|{description}"
    out, found = [], False
    for line in lines:
        if line.startswith("#") or not line.strip():
            out.append(line)
            continue
        parts = line.split("|")
        if len(parts) >= 3 and parts[0].strip() == role and parts[2].strip() == rel_path:
            out.append(new_line)
            found = True
        else:
            out.append(line)
    if not found:
        out.append(new_line)
    DATAFILE.write_text("\n".join(out) + "\n", encoding="utf-8")

def new_run_id():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]

def file_checksum(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

print("ROOT:", ROOT)
print(load_registry())


ROOT: E:\Amit\Project\Hospital project
           role    zone                                       path  \
0           raw  bronze                 data/raw/diabetic_data.csv   
1         clean  silver        data/lake/silver/encounters.parquet   
2      features    gold      data/lake/gold/model_features.parquet   
3     sequences    gold       data/lake/gold/rnn_sequences.parquet   
4   predictions    gold    data/lake/gold/test_predictions.parquet   
5       metrics    gold  data/lake/gold/experiment_results.parquet   
6          mart  export          data/exports/mart_readmission.csv   
7          mart  export        data/exports/mart_clinical_risk.csv   
8          mart  export    data/exports/mart_model_performance.csv   
9          mart  export         data/exports/mart_dq_scorecard.csv   
10     manifest     ops    data/lake/bronze/_manifests/latest.json   
11     champion     ops              models/champion_register.json   

                                      description 

## 2. Load silver encounters

Read conformed silver data produced by Phase 0.


In [2]:
from sqlalchemy import create_engine, text

silver_path = registry_path("clean", "data/lake/silver/encounters.parquet")
assert silver_path.exists(), "Run Phase 0 first"
df = pd.read_parquet(silver_path)
print("silver", df.shape)


silver (101766, 50)


## 3. Connect SQLAlchemy warehouse

Default is portable SQLite. Set `DATABASE_URL` for PostgreSQL/MySQL.


In [3]:
db_url = os.environ.get("DATABASE_URL", f"sqlite:///{(ROOT / 'data' / 'warehouse' / 'hospital.db').as_posix()}")
(ROOT / "data" / "warehouse").mkdir(parents=True, exist_ok=True)
engine = create_engine(db_url)
print("Engine:", db_url.split('://')[0])


Engine: sqlite


## 4. Build Kimball dim/fact tables and load

Patients, admissions, medications, lab results - grain documented in `docs/phase1_modeling_marts_sql.md`.


In [4]:
dim_patient = df[["patient_nbr", "race", "gender", "age"]].drop_duplicates("patient_nbr")
fact_admission = df[[
    "encounter_id", "patient_nbr", "admission_type_id", "discharge_disposition_id",
    "admission_source_id", "time_in_hospital", "payer_code", "medical_specialty",
    "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses",
    "diag_1", "diag_2", "diag_3", "change", "diabetesMed", "readmitted",
]].copy()
med_cols = [c for c in df.columns if c in {
    "metformin","repaglinide","nateglinide","chlorpropamide","glimepiride","acetohexamide",
    "glipizide","glyburide","tolbutamide","pioglitazone","rosiglitazone","acarbose","miglitol",
    "troglitazone","tolazamide","examide","citoglipton","insulin","glyburide-metformin",
    "glipizide-metformin","glimepiride-pioglitazone","metformin-rosiglitazone","metformin-pioglitazone"
}]
fact_medication = df[["encounter_id"] + med_cols].copy()
fact_lab = df[["encounter_id", "num_lab_procedures", "num_procedures", "num_medications", "max_glu_serum", "A1Cresult"]].copy()

dim_patient.to_sql("dim_patient", engine, if_exists="replace", index=False)
fact_admission.to_sql("fact_admission", engine, if_exists="replace", index=False)
fact_medication.to_sql("fact_medication", engine, if_exists="replace", index=False)
fact_lab.to_sql("fact_lab", engine, if_exists="replace", index=False)
print("Loaded modeling layer tables")
print("dim_patient", len(dim_patient), "fact_admission", len(fact_admission))


Loaded modeling layer tables
dim_patient 71518 fact_admission 101766


## 5. Enrich admissions for metrics

Adds `readmit_30d`, any-readmission proxy, and total visits used by the metric dictionary.


In [5]:
enrich_sql = """
CREATE TABLE fact_admission_enriched AS
SELECT a.*,
    CASE WHEN a.readmitted = '<30' THEN 1 ELSE 0 END AS readmit_30d,
    CASE WHEN a.readmitted IN ('<30','>30') THEN 1 ELSE 0 END AS readmit_any,
    (a.number_outpatient + a.number_emergency + a.number_inpatient) AS total_visits
FROM fact_admission a
"""
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS fact_admission_enriched"))
    conn.execute(text(enrich_sql))
print("fact_admission_enriched ready")


fact_admission_enriched ready


## 6. Run analytical SQL files (>=12)

Executes every `.sql` file in `sql/` (includes the seven brief-required queries).


In [6]:
sql_dir = ROOT / "sql"
results = {}
for sql_file in sorted(sql_dir.glob("*.sql")):
    q = sql_file.read_text(encoding="utf-8")
    try:
        res = pd.read_sql(q, engine)
        results[sql_file.name] = res
        print(sql_file.name, "->", res.shape)
    except Exception as e:
        print("SQL fail", sql_file.name, e)
        raise
print("SQL files executed:", len(results))


01_readmission_rate_by_hospital.sql -> (73, 3)


02_readmission_by_age_group.sql

 -> (10, 3)
03_readmission_by_diagnosis.sql -> (50, 3)
04_average_length_of_stay.sql -> (1, 3)
05_high_risk_patient_identification.sql -> (200, 5)


06_medication_usage_patterns.sql -> (4, 3)
07_frequent_visitors.sql -> (200, 4)


08_readmission_by_gender.sql -> (3, 3)
09_los_by_readmission.sql -> (3, 3)
10_lab_intensity.sql -> (2, 3)


11_admission_type_impact.sql -> (8, 4)
12_join_patient_lab_med.sql -> (100, 7)
SQL files executed: 12


## 7. Build and export `mart_readmission`

Certified BI mart: joins patients + admissions + active medication counts.


In [7]:
mart_sql = """
SELECT p.age, p.gender, p.race,
       a.admission_type_id, a.time_in_hospital, a.diag_1,
       a.number_inpatient, a.number_emergency, a.number_outpatient,
       a.total_visits, a.readmit_30d, a.readmit_any, a.readmitted,
       a.encounter_id, a.patient_nbr
FROM fact_admission_enriched a
JOIN dim_patient p ON a.patient_nbr = p.patient_nbr
"""
mart_readmission = pd.read_sql(mart_sql, engine)

med_long = fact_medication.melt(id_vars=["encounter_id"], var_name="medication", value_name="status")
med_active = med_long[med_long["status"].isin(["Steady", "Up", "Down"])].groupby("encounter_id").size().rename("active_med_count")
mart_readmission = mart_readmission.merge(med_active, on="encounter_id", how="left")
mart_readmission["active_med_count"] = mart_readmission["active_med_count"].fillna(0).astype(int)

exports = ROOT / "data" / "exports"
exports.mkdir(parents=True, exist_ok=True)
mart_readmission.to_csv(exports / "mart_readmission.csv", index=False)
mart_readmission.to_sql("mart_readmission", engine, if_exists="replace", index=False)
upsert_registry("mart", "export", "data/exports/mart_readmission.csv", "Certified BI mart - hospital/readmission KPIs")
print("mart_readmission rows:", len(mart_readmission))


mart_readmission rows: 101766


## 8. Metric dictionary, NoSQL seeds, vector docs

Single source of truth for KPI definitions; RAG documents for the chatbot.


In [8]:
metric_dict = {
    "readmission_rate_30d": "sum(readmit_30d)/count(encounters) for cohort",
    "avg_los": "avg(time_in_hospital)",
    "high_risk_rate": "share with predicted risk_band=High in scored mart",
    "frequent_visitor": "number_inpatient >= 2 OR total_visits >= 3",
}
(ROOT / "data" / "nosql" / "metric_dictionary.json").write_text(json.dumps(metric_dict, indent=2), encoding="utf-8")

nosql = ROOT / "data" / "nosql"
for name in ["chat_sessions.json", "audit_events.json"]:
    p = nosql / name
    if not p.exists():
        p.write_text("[]", encoding="utf-8")

docs = [
    {"id": "metric_readmit", "text": "Readmission rate 30d = sum(readmit_30d)/count(encounters). Primary outcome is readmission within 30 days."},
    {"id": "intended_use", "text": "Analytics decision-support only. Not a medical device. Not for standalone clinical decisions."},
    {"id": "bias", "text": "Model must be monitored for unfair performance differences by gender and age group."},
]
(nosql / "rag_documents.json").write_text(json.dumps(docs, indent=2), encoding="utf-8")

try:
    import chromadb
    client = chromadb.PersistentClient(path=str(ROOT / "data" / "vectordb"))
    col = client.get_or_create_collection("project_knowledge")
    col.upsert(ids=[d["id"] for d in docs], documents=[d["text"] for d in docs])
    print("Chroma vector DB seeded")
except Exception as e:
    print("Chroma optional seed skipped:", e)


Chroma vector DB seeded


## 9. Phase 1 summary


In [9]:
print("Phase 1 complete. Mart rows:", len(mart_readmission))
print("Sample SQL outputs:", {k: results[k].head(2).to_dict() for k in list(results)[:2]})


Phase 1 complete. Mart rows: 101766
Sample SQL outputs: {'01_readmission_rate_by_hospital.sql': {'medical_specialty': {0: '?', 1: 'InternalMedicine'}, 'encounters': {0: 49949, 1: 14635}, 'readmission_rate_30d': {0: 0.11573805281387015, 1: 0.11247010591048856}}, '02_readmission_by_age_group.sql': {'age': {0: '[0-10)', 1: '[10-20)'}, 'encounters': {0: 163, 1: 723}, 'readmission_rate_30d': {0: 0.018404907975460124, 1: 0.08160442600276625}}}
